In [ ]:
import django

django.setup()

In [ ]:
import logging
import sys

import pokebase as pb
from pokebase import APIMetadata, APIResource
from requests.exceptions import HTTPError

from typing import Any
from pprint import PrettyPrinter

from django.db import transaction, models

from pokedex.models import (
    Move,
    Pokemon,
    PokemonAbility,
    PokemonForm,
    PokemonFormType,
    PokemonMove,
    PokemonMoveVersion,
    PokemonSpecies,
    PokemonSpeciesDexEntry,
    PokemonSpeciesVariety,
    PokemonStat,
    PokemonType,
    VersionGameIndex,
)

logging.basicConfig(
    format="%(asctime)s [%(levelname)s] %(message)s",
    level=logging.DEBUG,
    stream=sys.stdout,
    force=True,
)


logger = logging.getLogger(__name__)


class PokemonBuilder:
    def __get_api_metadata_as_dict(self, api_metadata: APIMetadata) -> dict[str, Any]:
        _ALLOWED_TYPES = (bool, str, int, float, APIMetadata, APIResource)

        if type(api_metadata) is not APIMetadata:
            raise Exception(f"objeto informado deve implementar {APIMetadata}")

        result = {}

        for key, value in vars(api_metadata).items():
            if value and type(value) not in _ALLOWED_TYPES:
                logger.warning(
                    f"[{key}] value deve ser {_ALLOWED_TYPES}, mas recebeu {type(value)}"
                )
                continue

            if type(value) == APIMetadata:
                result[key] = self.__get_api_metadata_as_dict(value)
                continue

            result[key] = str(value) if type(value) else value

        return result

    def __get_object_or_none(
        self, model: models.Model, **lookup: dict[str, Any]
    ) -> models.Model | None:
        try:
            return model.objects.get(**lookup)
        except model.DoesNotExist:
            return None

    def __get_or_create_pokemon_from_resource(self) -> tuple[Pokemon | None, bool]:
        return Pokemon.objects.get_or_create(
            name=self.resource.name,
            defaults={
                "name": self.resource.name,
                "base_experience": self.resource.base_experience,
                "height": self.resource.height,
                "is_default": self.resource.is_default,
                "order": self.resource.order,
                "weight": self.resource.weight,
                "sprites": self.__get_api_metadata_as_dict(self.resource.sprites),
                "cries": self.__get_api_metadata_as_dict(self.resource.cries),
            },
        )

    def __get_evolves_from_species(self) -> PokemonSpecies | None:
        if not self.resource.species.evolves_from_species:
            return None

        return self.__get_object_or_none(
            PokemonSpecies, name=self.resource.species.evolves_from_species.name
        )

    def __link_abilities(self, pokemon: Pokemon):
        for a in self.resource.abilities:
            logger.debug("linking ability: %s" % a.ability)
            pokemon.abilities.add(
                PokemonAbility.objects.get_or_create(
                    slot=a.slot,
                    is_hidden=a.is_hidden,
                    ability=str(a.ability),
                )[0]
            )

    def __link_forms(self, pokemon: Pokemon):
        for f in self.resource.forms:
            logger.debug("linking form: %s" % f.name)
            form, _ = PokemonForm.objects.get_or_create(
                name=f.name,
                defaults={
                    "order": f.order,
                    "form_order": f.form_order,
                    "is_default": f.is_default,
                    "is_battle_only": f.is_battle_only,
                    "is_mega": f.is_mega,
                    "form_name": f.form_name,
                    "sprites": self.__get_api_metadata_as_dict(f.sprites),
                    "version_group": f.version_group,
                    "pokemon": pokemon,
                },
            )

            logger.debug("linking pokemon form types...")
            self.__link_pokemon_form_types(form, f.types)

    def __link_game_indices(self, pokemon: Pokemon):
        for i in self.resource.game_indices:
            logger.debug(
                "linking index %s#%d a %s"
                % (i.version.name, i.game_index, pokemon.name)
            )
            pokemon.game_indices.add(
                VersionGameIndex.objects.get_or_create(
                    game_index=i.game_index, version=i.version.name
                )[0]
            )

    def __link_moves(self, pokemon: Pokemon):
        p_moves = []

        for m in self.resource.moves:
            logger.debug("linking move: %s" % str(m.move))
            r_move = pb.move(str(m.move))
            p_move_versions = []

            move, _ = Move.objects.get_or_create(
                name=r_move.name,
                defaults={
                    "accuracy": r_move.accuracy,
                    "effect_chance": r_move.effect_chance,
                    "pp": r_move.pp,
                    "priority": r_move.priority,
                    "power": r_move.power,
                    "damage_class": r_move.damage_class,
                    "generation": r_move.generation,
                    "target": r_move.target,
                    "type": r_move.type,
                    "pokemon": pokemon,
                },
            )

            p_move, _ = PokemonMove.objects.get_or_create(move=move)

            for v in m.version_group_details:
                p_move_versions.append(
                    PokemonMoveVersion.objects.get_or_create(
                        move_learn_method=v.move_learn_method,
                        level_learned_at=v.level_learned_at,
                        version_group=v.version_group,
                        order=v.order,
                    )[0]
                )

            p_move.version_group_details.add(*p_move_versions)
            p_moves.append(p_move)

        pokemon.moves.add(*p_moves)

    def __link_pokemon_form_types(self, form: PokemonForm, r_types: APIResource):
        self.__link_types(PokemonFormType, form, r_types)

    def __link_pokemon_types(self, pokemon: Pokemon):
        self.__link_types(PokemonType, pokemon, self.resource.types)

    def __link_species_pokedex_numbers(self, species: PokemonSpecies):
        r_species = self.resource.species

        for pn in r_species.pokedex_numbers:
            logger.debug("linking pokedex: %s#%d" % (pn.entry_number, str(pn.pokedex)))
            species.pokedex_numbers.add(
                PokemonSpeciesDexEntry.objects.get_or_create(
                    entry_number=pn.entry_number,
                    pokedex=str(pn.pokedex),
                )[0]
            )

    def __link_species_varieties(self, species: PokemonSpecies):
        r_species = self.resource.species

        for sv in r_species.varieties:
            logger.debug("linking variety: %s" % str(sv.pokemon))
            sv_pokemon = self.__get_object_or_none(Pokemon, name=str(sv.pokemon))

            if not sv_pokemon and species.name != str(sv.pokemon):
                builder = PokemonBuilder(resource=str(sv.pokemon))
                sv_pokemon = builder.build(link_species=True)[0]

            species.varieties.add(
                PokemonSpeciesVariety.objects.get_or_create(
                    is_default=sv.is_default,
                    pokemon=sv_pokemon,
                )[0]
            )

    def __link_species(self, pokemon: Pokemon):
        if pokemon.species:
            logger.debug("%s species already linked!" % pokemon.name)
            return

        r_species = self.resource.species

        logger.debug("linking species: %s" % r_species.name)
        species, _ = PokemonSpecies.objects.update_or_create(
            name=r_species.name,
            defaults={
                "order": r_species.order,
                "gender_rate": r_species.gender_rate,
                "capture_rate": r_species.capture_rate,
                "base_happiness": r_species.base_happiness,
                "is_baby": r_species.is_baby,
                "is_legendary": r_species.is_legendary,
                "is_mythical": r_species.is_mythical,
                "hatch_counter": r_species.hatch_counter,
                "has_gender_differences": r_species.has_gender_differences,
                "forms_switchable": r_species.forms_switchable,
                "growth_rate": r_species.growth_rate,
                "egg_groups": [g.name for g in r_species.egg_groups],
                "color": r_species.color,
                "shape": r_species.shape,
                "evolves_from_species": self.__get_evolves_from_species(),
                "generation": r_species.generation,
            },
        )

        self.__link_species_pokedex_numbers(species)
        self.__link_species_varieties(species)

        pokemon.species = species
        pokemon.save()

    def __link_stats(self, pokemon: Pokemon):
        for s in self.resource.stats:
            logger.debug("linking base %s: %d" % (s.base_stat, s.stat))
            pokemon.stats.add(
                PokemonStat.objects.get_or_create(
                    stat=s.stat,
                    effort=s.effort,
                    base_stat=s.base_stat,
                )[0]
            )

    def __link_types(
        self,
        t_model: PokemonType | PokemonFormType,
        m_obj: Pokemon | PokemonForm,
        r_types: APIResource,
    ):
        for t in r_types:
            logger.debug("linking type: %s" % t.type)
            m_obj.types.add(
                t_model.objects.get_or_create(
                    slot=t.slot,
                    type=t.type,
                )[0]
            )

    @transaction.atomic()
    def build(self, **kwargs) -> tuple[Pokemon | None, bool]:
        logger.info("building Pokemon: %s..." % self.resource.name)
        pokemon, created = self.__get_or_create_pokemon_from_resource()

        if kwargs.get("link_abilities", False):
            logger.debug("linking abilities...")
            self.__link_abilities(pokemon)

        if kwargs.get("link_forms", False):
            logger.debug("linking forms...")
            self.__link_forms(pokemon)

        if kwargs.get("link_game_indices", False):
            logger.debug("linking game_indices...")
            self.__link_game_indices(pokemon)

        if kwargs.get("link_moves", False):
            logger.debug("linking moves...")
            self.__link_moves(pokemon)

        if kwargs.get("link_species", False):
            logger.debug("linking species...")
            self.__link_species(pokemon)

        if kwargs.get("link_stats", False):
            logger.debug("linking stats...")
            self.__link_stats(pokemon)

        if kwargs.get("link_types", False):
            logger.debug("linking types...")
            self.__link_pokemon_types(pokemon)

        return pokemon, created

    def __init__(self, *args, **kwargs):
        self.resource = pb.pokemon(kwargs.get("resource", "bulbasaur"))

In [ ]:
# offset = 1
# limit = 9

# print(f"\n--- Iniciando ciclo de busca (ID Inicial: {offset}) ---")

# fim_do_ciclo = offset + limit

# for index in range(offset, fim_do_ciclo):
#     try:
#         builder = PokemonBuilder(resource=index)
#         pokemon, created = builder.build(
#             # link_game_indices=True,
#             # link_abilities=True,
#             # link_species=True,
#             # link_stats=True,
#             # link_types=True,
#             # link_moves=True,
#             link_forms=True,
#         )
#     except HTTPError as e:
#         # Verifica se o erro é o 404 (Recurso não encontrado / Fim da Pokédex)
#         # Nota: se usar a lib requests, mude para: e.response.status_code == 404
#         status_code = getattr(
#             e, "code", getattr(getattr(e, "response", None), "status_code", None)
#         )

#         if status_code == 404:
#             print(
#                 f"\n[FIM] Erro HTTP 404 encontrado no ID {index}. Fim da Pokédex alcançado."
#             )
#             must_stop = True
#             break  # Sai do loop 'for' imediatamente
#         else:
#             # Se for outro erro HTTP (ex: 500, 503), avisa e continua tentando o próximo
#             print(f"Erro HTTP {status_code} no ID {index}, mas continuando...")

# offset = fim_do_ciclo